# Data Preparation: Caregiver Dataset

**ISSS623 Group Project - Topic C (caregiver angle) · Owner: Data Preparation Lead**

This notebook builds the dataset used by this project, starting from the raw CDC file:

| Step | Output | Rows |
|---|---|---|
| 1. Extract caregiver fields from raw fixed-width file | 4 columns | 457,670 |
| 2. Merge onto the course-supplied Topic C extract | `brfss2024_topic_c_caregiver.csv` (41 cols) | 457,670 |
| 3. Recode per the variable dictionary | recoded frame | 457,670 |
| 4. Cohort construction | `analytic_main.csv` | **86,676** |
| 5. Caregivers-only nested subset | `analytic_caregivers.csv` | **19,263** |
| 6. Stratified 70/30 train/test indices | `train_test_indices.csv` | 86,676 |

**Inputs required in the working folder** (`data/` relative to this notebook):
`LLCP2024ASC.zip` (CDC raw, 41.5 MB), `brfss2024_topic_c.csv` and `brfss2024_variable_dictionary.csv` (course repo).

Codes throughout: 7/77 = don't know, 9/99 = refused, 88 = "none" in day-counts, blank = not asked (skip logic).
Special: `CAREGIV1` code 8 = "expects to provide care within two years" (NOT "none"); `CHECKUP1` code 8 = "never" (a valid care gap).

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
DATA = Path('data')
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 50)
print(pd.__version__, np.__version__)

2.2.3 2.2.6


## Step 1: Extract caregiver fields from the raw fixed-width file

`CAREGIV1` and the caregiver-detail items sit in the **Caregiver optional module** (16 of 53 states in 2024) and are
not in the Topic C extract. We read them straight from the raw ASCII file at the column positions
given in the course variable dictionary. `read_fwf` colspecs are 0-indexed and end-exclusive, so a dictionary entry
`(col_start, col_end)` becomes `(col_start - 1, col_end)`.

In [ ]:
# Unzip the raw file if not already extracted
zip_path = DATA / 'LLCP2024ASC.zip'
asc_candidates = list(DATA.glob('LLCP2024.ASC*'))
if not asc_candidates:
    with zipfile.ZipFile(zip_path) as z:
        member = z.namelist()[0]
        z.extract(member, DATA)
    asc_candidates = list(DATA.glob('LLCP2024.ASC*'))
RAW = asc_candidates[0]
print('raw file:', repr(RAW.name), f'{RAW.stat().st_size/1e6:.0f} MB')

raw file: 'LLCP2024.ASC ' 944 MB


In [ ]:
dictionary = pd.read_csv(DATA / 'brfss2024_variable_dictionary.csv')
CAREGIVER_FIELDS = ['CAREGIV1', 'CRGVREL5', 'CRGVHRS2', 'CRGVLNG2']
ALIGN_FIELD = ['_STATE']

spec = dictionary[dictionary['variable'].isin(ALIGN_FIELD + CAREGIVER_FIELDS)]
print(spec[['variable', 'col_start', 'col_end']].to_string(index=False))

colspecs = [(int(r.col_start) - 1, int(r.col_end)) for r in spec.itertuples()]
names = spec['variable'].tolist()
raw_cg = pd.read_fwf(RAW, colspecs=colspecs, names=names, dtype=float)
assert len(raw_cg) == 457_670, f'expected 457,670 rows, got {len(raw_cg):,}'
print(f'rows read: {len(raw_cg):,}')

variable  col_start  col_end
  _STATE          1        2
CAREGIV1        325      325
CRGVREL5        326      327
CRGVHRS2        334      334
CRGVLNG2        335      335


rows read: 457,670


### Extraction validity checks

1. The `CAREGIV1` value distribution must match the official codebook code set exactly.
2. Caregiver prevalence should be ~1 in 5 (CDC's published figure).

In [4]:
dist = raw_cg['CAREGIV1'].value_counts(dropna=False).sort_index()
print(dist.to_string())
expected_codes = {1.0, 2.0, 7.0, 8.0, 9.0}
assert set(dist.index.dropna()) == expected_codes, 'unexpected CAREGIV1 codes - check column offsets'

asked = raw_cg['CAREGIV1'].notna()
prev = (raw_cg.loc[asked, 'CAREGIV1'] == 1).mean()
print(f'\nasked the module: {asked.sum():,} ({asked.mean()*100:.1f}% of file)')
print(f'raw caregiver share among asked: {prev*100:.1f}% (CDC benchmark ~20%)')
assert 0.15 < prev < 0.30

CAREGIV1
1.0     21137
2.0     74592
7.0       241
8.0       215
9.0       102
NaN    361383

asked the module: 96,287 (21.0% of file)
raw caregiver share among asked: 22.0% (CDC benchmark ~20%)


## Step 2: Merge with Topic C extract

BRFSS `SEQNO` is not unique across states, so we do a **positional** merge and *prove* it is safe by asserting the
state identifier matches row-for-row across both sources before concatenating.

In [5]:
topic_c = pd.read_csv(DATA / 'brfss2024_topic_c.csv')
assert len(topic_c) == len(raw_cg)
assert (topic_c['_STATE'].astype(float).values == raw_cg['_STATE'].values).all(), 'row order mismatch - do NOT merge'
print('row alignment verified on _STATE for all', f'{len(topic_c):,}', 'rows')

working = pd.concat([topic_c, raw_cg[CAREGIVER_FIELDS]], axis=1)
assert working.shape == (457_670, 41), working.shape
working.to_csv(OUT / 'brfss2024_topic_c_caregiver.csv', index=False)
print('written outputs/brfss2024_topic_c_caregiver.csv', working.shape)

row alignment verified on _STATE for all 457,670 rows


written outputs/brfss2024_topic_c_caregiver.csv (457670, 41)


## Step 3: Recode per the variable dictionary

Every recode below is **codebook-driven** (a fixed mapping from the CDC's published value labels), never estimated
from the data. Hence, recoding the full file before the train/test split cannot leak information.

Recoded columns get a `r_` prefix; originals are kept untouched for auditability.

In [ ]:
df = working.copy()

# primaryu outcome
ck = df['CHECKUP1'].replace({7: np.nan, 9: np.nan})
df['r_no_checkup'] = np.where(ck.isna(), np.nan, (ck != 1).astype(float))

# secondary outcome, i.e. cost barrier to seeing a doctor
df['r_cost_barrier'] = df['MEDCOST1'].replace({2: 0, 7: np.nan, 9: np.nan})

# key predictor - caregiver
df['r_caregiver'] = df['CAREGIV1'].replace({2: 0, 7: np.nan, 8: np.nan, 9: np.nan})

# access
ins_map = {1: 'employer', 2: 'private', 3: 'medicare', 4: 'medicare',
           5: 'medicaid_state', 6: 'medicaid_state', 9: 'medicaid_state',
           7: 'military_ihs', 8: 'military_ihs', 10: 'other_gov',
           88: 'uninsured', 77: 'not_reported', 99: 'not_reported'}
df['r_insurance'] = df['PRIMINS2'].map(ins_map)
df['r_persdoc'] = df['PERSDOC3'].replace({2: 1, 3: 0, 7: np.nan, 9: np.nan})

# health status
df['r_genhlth'] = df['GENHLTH'].replace({7: np.nan, 9: np.nan})
df['r_menthlth'] = df['MENTHLTH'].replace({88: 0, 77: np.nan, 99: np.nan})

# chronic conditions
df['r_diabetes'] = df['DIABETE4'].replace({2: 0, 3: 0, 4: 0, 7: np.nan, 9: np.nan})
df['r_michd'] = df['_MICHD'].replace({2: 0})

# disability
for src, dst in [('DIFFWALK', 'r_diffwalk'), ('DECIDE', 'r_decide'), ('DIFFALON', 'r_diffalon')]:
    df[dst] = df[src].replace({2: 0, 7: np.nan, 9: np.nan})

# behaviours
df['r_smoker'] = df['_SMOKER3'].map({1: 'current', 2: 'current', 3: 'former', 4: 'never'})  # 9 -> NaN
df['r_active'] = df['_TOTINDA'].replace({2: 0, 9: np.nan})

# demo
df['r_female'] = df['SEXVAR'].replace({1: 0, 2: 1})
df['r_ageband'] = df['_AGEG5YR'].replace({14: np.nan})
df['r_race'] = df['_RACEGR3'].replace({9: np.nan})
df['r_educ'] = df['_EDUCAG'].replace({9: np.nan})
df['r_income'] = df['_INCOMG1'].astype(float).map(
    lambda v: 'not_reported' if v == 9 else (f'band_{int(v)}' if v in range(1, 8) else np.nan))
emp_map = {1: 'employed', 2: 'employed', 3: 'unemployed', 4: 'unemployed',
           5: 'homemaker_student', 6: 'homemaker_student', 7: 'retired', 8: 'unable'}
df['r_employ'] = df['EMPLOY1'].map(emp_map)

PREDICTORS = ['r_caregiver', 'r_insurance', 'r_persdoc', 'r_cost_barrier',
              'r_genhlth', 'r_menthlth', 'r_diabetes', 'r_michd',
              'r_diffwalk', 'r_decide', 'r_diffalon', 'r_smoker', 'r_active',
              'r_female', 'r_ageband', 'r_race', 'r_educ', 'r_income', 'r_employ']
print(len(PREDICTORS), 'predictors (MEDCOST1/r_cost_barrier doubles as the secondary outcome)')

19 predictors (MEDCOST1/r_cost_barrier doubles as the secondary outcome)


## Step 4: Cohort construction

```
457,670  full 2024 file
-> keep rows asked CAREGIV1 
 96,287  module sample
 -> valid caregiver status AND valid outcome
 94,670
 -> complete cases on all predictors 
 86,676  MAIN ANALYTIC SAMPLE
```
`r_insurance` retains `not_reported` (77/99) and `r_income` retains `not_reported` (code 9, 16.65% of the module
sample) as explicit categories - deleting either would discard data non-randomly. All other predictors are
complete-case (<2% invalid each within the module sample).

In [7]:
flow = [('full 2024 file', len(df))]

module = df[df['CAREGIV1'].notna()].copy()
flow.append(('asked CAREGIV1 (16 module states)', len(module)))

valid = module[module['r_caregiver'].notna() & module['r_no_checkup'].notna()].copy()
flow.append(('valid caregiver status + outcome', len(valid)))

main = valid.dropna(subset=PREDICTORS).copy()
flow.append(('MAIN ANALYTIC SAMPLE (complete cases)', len(main)))

for label, n in flow:
    print(f'{n:>9,}  {label}')

assert len(module) == 96_287
assert len(valid) == 94_670
assert len(main) == 86_676, f'expected 86,676 - got {len(main):,}'

  457,670  full 2024 file
   96,287  asked CAREGIV1 (16 module states)
   94,670  valid caregiver status + outcome
   86,676  MAIN ANALYTIC SAMPLE (complete cases)


In [ ]:
# characteristics
ev = main['r_no_checkup'].mean()
cg_share = main['r_caregiver'].mean()
gap_by_cg = main.groupby('r_caregiver')['r_no_checkup'].mean() * 100
print(f'event rate (no check-up): {ev*100:.1f}%')
print(f'caregivers: {cg_share*100:.1f}% ({int(main.r_caregiver.sum()):,})')
print(f'gap rate  non-caregivers: {gap_by_cg[0]:.1f}%  caregivers: {gap_by_cg[1]:.1f}%')

assert abs(ev - 0.182) < 0.005 and abs(cg_share - 0.222) < 0.005

keep_cols = ['_STATE'] + PREDICTORS + ['r_no_checkup'] + CAREGIVER_FIELDS[1:]
main[keep_cols].to_csv(OUT / 'analytic_main.csv', index=False)
print('\nwritten outputs/analytic_main.csv', main[keep_cols].shape)

event rate (no check-up): 18.2%
caregivers: 22.2% (19,263)
gap rate  non-caregivers: 19.1%  caregivers: 15.0%



written outputs/analytic_main.csv (86676, 24)


### Missingness table

Structural module non-participation (79% of the full file - never asked) is reported separately from
item-level missingness *within* the module sample; they are different phenomena.

In [9]:
rows = [('CAREGIV1 module', 'sampling frame',
         len(df) - len(module), (len(df) - len(module)) / len(df) * 100, 'Not asked (state did not field module) - defines the frame')]
recode_of = {'r_no_checkup': 'CHECKUP1', 'r_cost_barrier': 'MEDCOST1', 'r_caregiver': 'CAREGIV1'}
for col in ['r_no_checkup', 'r_cost_barrier'] + PREDICTORS:
    bad = module[col].isna().sum()
    decision = ('kept as category' if col in ('r_insurance', 'r_income') and bad == 0
                else 'exclude row' if bad else '-')
    rows.append((recode_of.get(col, col), 'outcome' if 'checkup' in col or col == 'r_cost_barrier' else 'predictor',
                 bad, bad / len(module) * 100, decision))
miss = pd.DataFrame(rows, columns=['variable', 'role', 'invalid_n', 'invalid_pct', 'decision'])
miss['invalid_pct'] = miss['invalid_pct'].round(2)
miss.to_csv(OUT / 'missingness_table.csv', index=False)
miss

,variable,role,invalid_n,invalid_pct,decision
0,CAREGIV1 module,sampling frame,361383,78.96,Not asked (state did not field module) - defin...
1,CHECKUP1,outcome,1080,1.12,exclude row
2,MEDCOST1,outcome,274,0.28,exclude row
3,CAREGIV1,predictor,558,0.58,exclude row
4,r_insurance,predictor,0,0.00,kept as category
5,r_persdoc,predictor,838,0.87,exclude row
6,MEDCOST1,outcome,274,0.28,exclude row
7,r_genhlth,predictor,256,0.27,exclude row
8,r_menthlth,predictor,1620,1.68,exclude row
9,r_diabetes,predictor,186,0.19,exclude row


## Step 5: Caregivers-only subset

The detail items (`CRGVREL5` relationship, `CRGVHRS2` weekly hours, `CRGVLNG2` duration) were asked **only of
confirmed caregivers** - their skip-logic missingness perfectly reconstructs `CAREGIV1`, so they are banned from
all pooled models and live only here, in the caregivers-only descriptive subset.

```
 21,137  caregivers in the module sample (CAREGIV1 = 1)
 20,976  ... with a valid check-up outcome
 19,263  ... complete cases on the main predictor set
```

In [ ]:
caregivers = main[main['r_caregiver'] == 1].copy()
print(f'caregiver complete cases: {len(caregivers):,}')
assert len(caregivers) == 19_263, f'expected 19,263 - got {len(caregivers):,}'

events = int(caregivers['r_no_checkup'].sum())
print(f'gap events among caregivers: {events:,} ({events/len(caregivers)*100:.1f}%)')

for c in ['CRGVREL5', 'CRGVHRS2', 'CRGVLNG2']:
    print(f'{c}: {caregivers[c].notna().mean()*100:.1f}% valid')

caregivers.to_csv(OUT / 'analytic_caregivers.csv', index=False)
print('\nwritten outputs/analytic_caregivers.csv', caregivers.shape)

caregiver complete cases: 19,263
gap events among caregivers: 2,890 (15.0%)
CRGVREL5: 99.9% valid
CRGVHRS2: 98.8% valid
CRGVLNG2: 98.5% valid



written outputs/analytic_caregivers.csv (19263, 61)


In [ ]:
hours_lbl = {1: '0-8 h', 2: '9-19 h', 3: '20-39 h', 4: '40+ h'}
sub = caregivers[caregivers['CRGVHRS2'].isin(hours_lbl)]
prof = (sub.assign(hours=sub['CRGVHRS2'].map(hours_lbl))
           .groupby('hours')['r_no_checkup'].agg(gap_rate='mean', n='count'))
prof['gap_rate'] = (prof['gap_rate'] * 100).round(1)
prof

,gap_rate,n
hours,,
0-8 h,14.8,13005
20-39 h,15.2,3089
9-19 h,16.2,2488


## Step 6: Stratified 70/30 train/test split

One split, stratified on the outcome, fixed seed - created **here**, before any modelling, and saved so every
team member fits and evaluates on identical partitions. The test set is to be evaluated **once**, for the final
comparison table.

In [12]:
from sklearn.model_selection import train_test_split

idx_train, idx_test = train_test_split(
    main.index, test_size=0.30, stratify=main['r_no_checkup'], random_state=RANDOM_STATE)

split = pd.Series('train', index=main.index, name='split')
split.loc[idx_test] = 'test'
split.rename_axis('row_index').to_frame().to_csv(OUT / 'train_test_indices.csv')

summary = main.groupby(split)['r_no_checkup'].agg(n='count', event_rate='mean')
summary['event_rate'] = (summary['event_rate'] * 100).round(2)
print(summary)
assert abs(summary.loc['train', 'event_rate'] - summary.loc['test', 'event_rate']) < 0.5
print('\nwritten outputs/train_test_indices.csv - evaluate the test partition exactly once.')

           n  event_rate
split                   
test   26003       18.17
train  60673       18.18

written outputs/train_test_indices.csv - evaluate the test partition exactly once.


## Outputs produced

| File | Contents |
|---|---|
| `outputs/brfss2024_topic_c_caregiver.csv` | 41-column working dataset (457,670 rows) |
| `outputs/analytic_main.csv` | main analytic sample, recoded (86,676) |
| `outputs/analytic_caregivers.csv` | caregivers-only nested subset (19,263) |
| `outputs/missingness_table.csv` | per-variable missingness + decisions |
| `outputs/train_test_indices.csv` | fixed stratified 70/30 partition labels |